# Badini GEC — Scaled-up ByT5-base Training

Trains **ByT5-base** on **60,000 synthetic pairs** with proper validation and
best-model selection. This is the 'serious' training run (vs the earlier
8k ByT5-small pilot).

**Runtime: Colab Pro → L4 GPU + High-RAM.** Runtime → Change runtime type → L4 GPU.

Expect ~1–3 hours depending on GPU. The notebook saves the best checkpoint to Drive.

In [ ]:
# 1. Clone repo + install
!git clone https://github.com/omarGH99/om_Project_Based_Course_II.git
%cd om_Project_Based_Course_II
!pip -q install transformers datasets accelerate sentencepiece

In [ ]:
# 2. Load the LARGE pairs (60k train / 800 held-out test)
def load_pairs(path):
    rows = [l.rstrip('\n').split('\t') for l in open(path, encoding='utf-8')]
    return [r for r in rows if len(r) >= 2]

train_rows = load_pairs('data/pairs_train_large.tsv')
test_rows  = load_pairs('data/pairs_test_large.tsv')
print('train', len(train_rows), 'test', len(test_rows))

In [ ]:
# 3. Tokenize
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL = 'google/byt5-base'        # bigger than small; needs a real GPU (L4 is fine)
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)

ds = Dataset.from_dict({'src':[r[0] for r in train_rows], 'tgt':[r[1] for r in train_rows]})
ds = ds.train_test_split(test_size=0.05, seed=13)   # 5% internal validation

def prep(b):
    x = tok(b['src'], max_length=256, truncation=True)
    y = tok(text_target=b['tgt'], max_length=256, truncation=True)
    x['labels'] = y['input_ids']; return x
ds = ds.map(prep, batched=True, remove_columns=['src','tgt'])
print(ds)

In [ ]:
# 4. Train with best-model selection
from transformers import (DataCollatorForSeq2Seq, Seq2SeqTrainer,
                          Seq2SeqTrainingArguments)

args = Seq2SeqTrainingArguments(
    output_dir='byt5-base-badini',
    per_device_train_batch_size=8,        # lower to 4 if you hit out-of-memory
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,        # effective batch = 32
    learning_rate=3e-4,
    num_train_epochs=3,                   # 3 epochs; best epoch is auto-kept below
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=1,                   # keep only the best (saves disk)
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    fp16=True,                            # faster; works on L4/T4
    logging_steps=100,
    report_to='none',
)
trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=ds['train'], eval_dataset=ds['test'],
    data_collator=DataCollatorForSeq2Seq(tok, model=model))
trainer.train()

In [ ]:
# 5. Quick correction test
model.eval()
def correct(text):
    ids = tok(text, return_tensors='pt', truncation=True, max_length=256).input_ids.to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_length=256, num_beams=4)
    return tok.decode(out[0], skip_special_tokens=True)

for r in test_rows[:5]:
    print('IN :', r[0])
    print('OUT:', correct(r[0]))
    print('REF:', r[1]); print()

In [ ]:
# 6. Evaluate on the 800 held-out test pairs (P / R / F0.5 / GLEU / EM)
import difflib
from collections import Counter
from tqdm import tqdm

def edits(s,t):
    sm=difflib.SequenceMatcher(a=s,b=t,autojunk=False); ops=set()
    for tag,i1,i2,j1,j2 in sm.get_opcodes():
        if tag!='equal': ops.add((i1,i2,tuple(t[j1:j2])))
    return ops
def prf(src,hyp,ref,beta=0.5):
    tp=fp=fn=0
    for s,h,r in zip(src,hyp,ref):
        s,h,r=s.split(),h.split(),r.split(); he,re_=edits(s,h),edits(s,r)
        tp+=len(he&re_); fp+=len(he-re_); fn+=len(re_-he)
    p=tp/(tp+fp) if tp+fp else 1.0; rc=tp/(tp+fn) if tp+fn else 1.0
    f=((1+beta**2)*p*rc/(beta**2*p+rc)) if p+rc else 0.0; return p,rc,f
def gleu(hyp,ref,n=4):
    num=den=0
    for h,r in zip(hyp,ref):
        h,r=h.split(),r.split()
        for k in range(1,n+1):
            hg=Counter(tuple(h[i:i+k]) for i in range(len(h)-k+1))
            rg=Counter(tuple(r[i:i+k]) for i in range(len(r)-k+1))
            num+=sum((hg&rg).values()); den+=max(sum(hg.values()),1)
    return num/max(den,1)

src=[r[0] for r in test_rows]; ref=[r[1] for r in test_rows]
hyp=[correct(s) for s in tqdm(src, desc='ByT5-base')]
p,rc,f=prf(src,hyp,ref); g=gleu(hyp,ref); em=sum(h==r for h,r in zip(hyp,ref))/len(ref)
print(f'\nByT5-base | P {p:.3f}  R {rc:.3f}  F0.5 {f:.3f}  GLEU {g:.3f}  EM {em:.3f}')

import pandas as pd
pd.DataFrame({'source':src,'reference':ref,'byt5_base':hyp}).to_csv('data/byt5_base_outputs.csv', index=False)
pd.DataFrame([dict(model='ByT5-base',precision=p,recall=rc,f05=f,gleu=g,em=em)]).to_csv('data/byt5_base_metrics.csv', index=False)
print('saved metrics + outputs to data/')

In [ ]:
# 7. Save the best model to Drive
from google.colab import drive; drive.mount('/content/drive')
trainer.save_model('/content/drive/MyDrive/byt5-base-badini')
tok.save_pretrained('/content/drive/MyDrive/byt5-base-badini')
print('saved ByT5-base to Drive')